# 📌 MODUL 04 — Pertemuan 05: Relational Join, Window Functions & Spark SQL
> **Tujuan Pembelajaran:** Menguasai penggabungan multi-tabel (**Join**), menerapkan **Window Functions** untuk perankingan Top-N per kelompok, memahami perbedaan `row_number()` vs `rank()`, dan mengeksekusi kueri deklaratif via **Spark SQL**.

## ⚙️ 1. Setup Environment & Inisialisasi SparkSession

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Responsi-Pertemuan05") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession Aktif!")

## 📊 2. Menyiapkan Dataset Sintetis
Kita siapkan 2 tabel:
1. **Tabel Transaksi:** Catatan penjualan di berbagai cabang kota.
2. **Tabel Target Cabang:** Target penjualan bulanan dan nama PIC per kota.

In [ ]:
# Tabel 1: Transaksi
transaksi_data = [
    ("TRX-101", "Magelang", "Elektronik", 3, 500000),
    ("TRX-102", "Magelang", "Fashion", 6, 120000),
    ("TRX-103", "Yogyakarta", "Elektronik", 2, 1200000),
    ("TRX-104", "Yogyakarta", "Makanan", 18, 25000),
    ("TRX-105", "Semarang", "Elektronik", 4, 750000),
    ("TRX-106", "Semarang", "Rumah Tangga", 5, 80000),
    ("TRX-107", "Purworejo", "Makanan", 25, 20000),
    ("TRX-108", "Purworejo", "Kesehatan", 8, 90000),
    ("TRX-109", "Magelang", "Kesehatan", 4, 150000),
    ("TRX-110", "Solo", "Fashion", 7, 110000),
]
df_transaksi = spark.createDataFrame(
    transaksi_data, 
    ["order_id", "kota", "kategori", "unit_terjual", "harga_satuan"]
).withColumn("pendapatan", F.col("unit_terjual") * F.col("harga_satuan"))

# Tabel 2: Master Target Cabang
target_data = [
    ("Magelang", "Rani", 2500000),
    ("Yogyakarta", "Joko", 3500000),
    ("Semarang", "Sari", 3000000),
    ("Solo", "Bayu", 1500000),
    ("Purworejo", "Fitri", 1000000),
]
df_target = spark.createDataFrame(target_data, ["kota", "pic_cabang", "target_bulanan"])

print("Tabel Transaksi:")
df_transaksi.show(5)
print("Tabel Target Cabang:")
df_target.show()

## 🔗 3. Operasi Join: Best Practice "Agregasi Dulu Baru Join!"
**Konsep Kritis:** Daripada menggabungkan ribuan baris transaksi langsung ke tabel target (menyebabkan *network shuffling* besar), ringkaslah tabel transaksi per kota terlebih dahulu, baru kemudian di-join dengan tabel target!

In [ ]:
# 1. Agregasi total pendapatan per kota dari transaksi
df_omset_kota = df_transaksi.groupBy("kota").agg(
    F.sum("pendapatan").alias("total_realisasi")
)

# 2. Inner join dengan tabel target berdasarkan kolom 'kota'
df_pencapaian = df_omset_kota.join(df_target, on="kota", how="inner") \
    .withColumn(
        "pencapaian_persen",
        F.round((F.col("total_realisasi") / F.col("target_bulanan")) * 100, 2)
    ) \
    .orderBy(F.col("pencapaian_persen").desc())

print("Hasil Analisis Pencapaian Target Cabang:")
df_pencapaian.show()

## 🪟 4. Window Functions: Menemukan Juara 1 (Top-1) per Kota

### Mengapa Window Function?
`groupBy()` akan memadatkan baris menjadi 1 per kota. Sedangkan **Window Function** menghitung peringkat tanpa membuang detail baris!

### Perbedaan `row_number()`, `rank()`, dan `dense_rank()`:
- `row_number()`: 1, 2, 3, 4 (nomor urut selalu unik, paling ideal untuk Top-N pasti).
- `rank()`: 1, 2, 2, 4 (ada lompatan urutan jika seri).
- `dense_rank()`: 1, 2, 2, 3 (tanpa lompatan jika seri).

In [ ]:
# Skenario: Cari Kategori Produk Terlaris (Juara 1) di Masing-masing Kota

# Langkah A: Ringkas omset per kombinasi kota dan kategori
df_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    F.sum("pendapatan").alias("total_omset_kategori")
)

# Langkah B: Definisikan window partisi per kota urut omset menurun
window_kota = Window.partitionBy("kota").orderBy(F.col("total_omset_kategori").desc())

# Langkah C: Terapkan row_number() dan saring rank == 1
df_kategori_terlaris = df_kota_kategori \
    .withColumn("peringkat", F.row_number().over(window_kota)) \
    .filter(F.col("peringkat") == 1) \
    .orderBy(F.col("total_omset_kategori").desc())

print("Kategori Terlaris (Juara 1) di Setiap Kota:")
df_kategori_terlaris.show()

## 🏛️ 5. Spark SQL: Menulis Kueri Relasional Murni
Kita dapat mengubah DataFrame menjadi tabel virtual (Temporary View) dan menjalankan query SQL murni dengan performa yang sama cepatnya:

In [ ]:
# 1. Daftarkan DataFrame sebagai Temporary View
df_transaksi.createOrReplaceTempView("tabel_transaksi")
df_target.createOrReplaceTempView("tabel_target")

# 2. Jalankan Kueri SQL Murni
df_sql = spark.sql("""
    SELECT 
        t.kota,
        tg.pic_cabang,
        COUNT(t.order_id) AS total_transaksi,
        SUM(t.pendapatan) AS total_omset
    FROM tabel_transaksi t
    JOIN tabel_target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY total_omset DESC
""")

print("Hasil Analisis Menggunakan Spark SQL:")
df_sql.show()

## 🛑 6. Menutup SparkSession

In [ ]:
spark.stop()
print("🛑 SparkSession ditutup dan resource sistem dibebaskan.")